In [5]:

import pandas as pd
from pathlib import Path

# Load the customer reviews dataset
file_path = Path("review.csv")

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())


Dataset shape: (6000, 16)

Column names:
['Customer_ID', 'review_text', 'sentiment_label', 'satisfaction_score', 'preferred_category', 'preferred_channel', 'customer_value_segment', 'customer_tenure_months', 'orders_last_12m', 'days_since_last_purchase', 'support_tickets_12m', 'return_rate', 'churn', 'purchase_next_30_days', 'data_origin', 'split']


In [12]:
df.head()

,Customer_ID,review_text,sentiment_label,satisfaction_score,preferred_category,preferred_channel,customer_value_segment,customer_tenure_months,orders_last_12m,days_since_last_purchase,support_tickets_12m,return_rate,churn,purchase_next_30_days,data_origin,split
0,CUST009609,I am unhappy with my Sports & Fitness order an...,negative,2.00,Sports & Fitness,Website,Low Value,26,8,84,4,0.190,1,0,synthetic_generated_for_training,train
1,CUST004601,I was disappointed with my Electronics purchas...,negative,1.56,Electronics,Mobile App,Medium Value,10,7,63,4,0.000,1,1,synthetic_generated_for_training,test
2,CUST000722,I was disappointed with my Sports & Fitness pu...,negative,1.89,Sports & Fitness,Website,Low Value,12,6,73,6,0.119,1,0,synthetic_generated_for_training,train
3,CUST007179,The Books shopping experience needs improvemen...,negative,2.41,Books,Physical Store,Low Value,12,4,322,0,0.210,1,0,synthetic_generated_for_training,train
4,CUST002160,"My Sports & Fitness order was frustrating, and...",negative,2.46,Sports & Fitness,Marketplace,Medium Value,20,6,36,2,0.327,0,1,synthetic_generated_for_training,validation


In [6]:
print("Dataset Shape:",df.shape)
print("\nColumn Names:")
print(df.columns.tolist())

Dataset Shape: (6000, 16)

Column Names:
['Customer_ID', 'review_text', 'sentiment_label', 'satisfaction_score', 'preferred_category', 'preferred_channel', 'customer_value_segment', 'customer_tenure_months', 'orders_last_12m', 'days_since_last_purchase', 'support_tickets_12m', 'return_rate', 'churn', 'purchase_next_30_days', 'data_origin', 'split']


In [7]:
print("\nFrist five reviews:")
display(df[["review_text","sentiment_label","split"]].head())


Frist five reviews:


,review_text,sentiment_label,split
0,I am unhappy with my Sports & Fitness order an...,negative,train
1,I was disappointed with my Electronics purchas...,negative,test
2,I was disappointed with my Sports & Fitness pu...,negative,train
3,The Books shopping experience needs improvemen...,negative,train
4,"My Sports & Fitness order was frustrating, and...",negative,validation


In [8]:
print("Sentiment Distribution:")
print(df["sentiment_label"].value_counts())

Sentiment Distribution:
sentiment_label
positive    3287
neutral     1988
negative     725
Name: count, dtype: int64


In [9]:
print("\nMissing review text:", df["review_text"].isna().sum())
print("Missing sentiment labels", df["sentiment_label"].isna().sum())


Missing review text: 0
Missing sentiment labels 0


In [10]:
from transformers import pipeline

MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"

sentiment_analyzer = pipeline(
    task="sentiment-analysis",
    model=MODEL_NAME
)

print("Sentiment model loaded successfully!")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 8629.64it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Sentiment model loaded successfully!


In [11]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Select held-out test data only
test_df = df.loc[
    df["split"].str.lower() == "test",
    ["review_text", "sentiment_label"]
].dropna().copy()

print("Test records:", len(test_df))

# Predict in batches
predictions = sentiment_analyzer(
    test_df["review_text"].astype(str).tolist(),
    batch_size=16,
    truncation=True
)

Test records: 600
